In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Supervised Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ---------------------------------------------------
# Load Dataset
# ---------------------------------------------------

las_old = pd.read_csv("WellA.csv")

las_old.columns = [
    'Depth','Gamma-ray','Shale_volume','Restivity',
    'Delta T','Vp','Vs','density',
    'density_calculated','Neutron Porosity',
    'Density_porosity','Poissons_ratio',
    'classification'
]

# ---------------------------------------------------
# Features and Target
# ---------------------------------------------------

X = las_old.drop("classification", axis=1)
y = las_old["classification"]

# ---------------------------------------------------
# Train-Test Split
# ---------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------
# Feature Scaling
# ---------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------
# Store Results
# ---------------------------------------------------

results = []

# ===================================================
# Logistic Regression
# ===================================================

print("="*60)
print("LOGISTIC REGRESSION")
print("="*60)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

lr_acc = accuracy_score(y_test, lr_pred)
lr_pre = precision_score(y_test, lr_pred, average='weighted')
lr_rec = recall_score(y_test, lr_pred, average='weighted')
lr_f1 = f1_score(y_test, lr_pred, average='weighted')

print("Accuracy :", lr_acc)
print("Precision:", lr_pre)
print("Recall   :", lr_rec)
print("F1 Score :", lr_f1)

results.append([
    "Logistic Regression",
    lr_acc,
    lr_pre,
    lr_rec,
    lr_f1
])

cm = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ===================================================
# KNN
# ===================================================

print("="*60)
print("KNN")
print("="*60)

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

knn_acc = accuracy_score(y_test, knn_pred)
knn_pre = precision_score(y_test, knn_pred, average='weighted')
knn_rec = recall_score(y_test, knn_pred, average='weighted')
knn_f1 = f1_score(y_test, knn_pred, average='weighted')

print("Accuracy :", knn_acc)
print("Precision:", knn_pre)
print("Recall   :", knn_rec)
print("F1 Score :", knn_f1)

results.append([
    "KNN",
    knn_acc,
    knn_pre,
    knn_rec,
    knn_f1
])

cm = confusion_matrix(y_test, knn_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap="Greens")
plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ===================================================
# Random Forest
# ===================================================

print("="*60)
print("RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Random Forest does not require scaling
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_pre = precision_score(y_test, rf_pred, average='weighted')
rf_rec = recall_score(y_test, rf_pred, average='weighted')
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print("Accuracy :", rf_acc)
print("Precision:", rf_pre)
print("Recall   :", rf_rec)
print("F1 Score :", rf_f1)

results.append([
    "Random Forest",
    rf_acc,
    rf_pre,
    rf_rec,
    rf_f1
])

cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap="Purples")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# ===================================================
# Comparison Table
# ===================================================

comparison = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n")
print("="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison)

# ===================================================
# Accuracy Comparison Graph
# ===================================================

plt.figure(figsize=(7,5))

sns.barplot(
    data=comparison,
    x="Algorithm",
    y="Accuracy"
)

plt.ylim(0,1)
plt.title("Classification Algorithm Comparison")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ====================================================
# LOAD DATA
# ====================================================

df = pd.read_csv("WellB.csv")

print("="*60)
print("Original Shape :", df.shape)
print("="*60)

# ====================================================
# REMOVE COLUMNS HAVING >50% MISSING VALUES
# ====================================================

threshold = len(df) * 0.5
df = df.dropna(axis=1, thresh=threshold)

print("After removing sparse columns:", df.shape)

# ====================================================
# FILL MISSING VALUES
# ====================================================

numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# ====================================================
# ENCODE CATEGORICAL COLUMN
# ====================================================

if "FLOW_KIND" in df.columns:
    df = pd.get_dummies(df, columns=["FLOW_KIND"], drop_first=True)

# ====================================================
# KEEP ONLY NUMERIC DATA
# ====================================================

numeric_df = df.select_dtypes(include=np.number)

print("\nNumeric Columns")
print(numeric_df.columns.tolist())

# ====================================================
# FEATURES & TARGET
# ====================================================

y = numeric_df["BORE_OIL_VOL"]
X = numeric_df.drop(columns=["BORE_OIL_VOL"])

# ====================================================
# TRAIN TEST SPLIT
# ====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ====================================================
# STANDARDIZATION
# ====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ====================================================
# RESULTS LIST
# ====================================================

results = []

# ====================================================
# LINEAR REGRESSION
# ====================================================

print("\n")
print("="*60)
print("LINEAR REGRESSION")
print("="*60)

lr = LinearRegression()

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_mse = mean_squared_error(y_test, lr_pred)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_pred)

print("MAE :", lr_mae)
print("MSE :", lr_mse)
print("RMSE:", lr_rmse)
print("R² Score :", lr_r2)

results.append([
    "Linear Regression",
    lr_mae,
    lr_rmse,
    lr_r2
])

plt.figure(figsize=(7,6))
plt.scatter(y_test, lr_pred, alpha=0.5)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)
plt.title("Linear Regression")
plt.xlabel("Actual Oil Volume")
plt.ylabel("Predicted Oil Volume")
plt.grid(True)
plt.show()

# ====================================================
# RANDOM FOREST REGRESSION
# ====================================================

print("\n")
print("="*60)
print("RANDOM FOREST REGRESSION")
print("="*60)

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Tree models don't require scaling
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("MAE :", rf_mae)
print("MSE :", rf_mse)
print("RMSE:", rf_rmse)
print("R² Score :", rf_r2)

results.append([
    "Random Forest",
    rf_mae,
    rf_rmse,
    rf_r2
])

plt.figure(figsize=(7,6))
plt.scatter(y_test, rf_pred, alpha=0.5)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)
plt.title("Random Forest Regression")
plt.xlabel("Actual Oil Volume")
plt.ylabel("Predicted Oil Volume")
plt.grid(True)
plt.show()

# ====================================================
# COMPARISON TABLE
# ====================================================

comparison = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "MAE",
        "RMSE",
        "R² Score"
    ]
)

comparison = comparison.sort_values(
    by="R² Score",
    ascending=False
)

print("\n")
print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison)

# ====================================================
# BAR CHART
# ====================================================

plt.figure(figsize=(6,5))

plt.bar(comparison["Algorithm"], comparison["R² Score"])

plt.ylabel("R² Score")
plt.title("Regression Model Comparison")
plt.ylim(0, 1)

plt.show()